In [1]:
# 1. Clone the repository and navigate into it
!git clone https://github.com/Ragu-123/indicf5.git
%cd indicf5

# 2. Install main F5-TTS packages in default environment (Enforce compatible transformers version)
!pip install -e .
!pip install numpy==2.0.0
!pip install "transformers<4.50.0,>=4.40.0"

# 3. Create a separate isolated virtual environment for Cadence to prevent version conflicts
!python -m venv /kaggle/working/cadence_env --without-pip
!curl -sSL https://bootstrap.pypa.io/get-pip.py -o get-pip.py
!/kaggle/working/cadence_env/bin/python get-pip.py
!rm get-pip.py

# 4. Install cadence-punctuation and pinned transformers inside the isolated environment in a single call
# This guarantees pip resolves to v4.51.3 instead of pulling the breaking v5.x
!/kaggle/working/cadence_env/bin/pip install "transformers==4.51.3" "cadence-punctuation"


Cloning into 'indicf5'...
remote: Enumerating objects: 168, done.
remote: Counting objects: 100% (168/168), done.
remote: Compressing objects: 100% (113/113), done.
remote: Total 168 (delta 82), reused 140 (delta 54), pack-reused 0 (from 0)
Receiving objects: 100% (168/168), 3.68 MiB | 28.56 MiB/s, done.
Resolving deltas: 100% (82/82), done.
/kaggle/working/indicf5
Obtaining file:///kaggle/working/indicf5
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.7/104.7 kB 5.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 62.5 MB/s eta 0:00:00a 0:00:01
  Preparing metadata (setup.py) ... done
   ━━━━━━

In [11]:
!python src/indicf5_finetune/merge_lora.py \
    --checkpoint-path /kaggle/input/datasets/ragunathravi/lora-finetuned-indicf5/indicf5/checkpoints/model_last.pt \
    --output-path /kaggle/working/model_last.pt

Loading checkpoint from /kaggle/input/datasets/ragunathravi/lora-finetuned-indicf5/indicf5/checkpoints/model_last.pt...
Merging in 'ema_model_state_dict'...
Merging in 'model_state_dict'...
Successfully merged 176 LoRA projection layers!
Saving merged checkpoint to /kaggle/working/model_last.pt...
SUCCESS: Merging complete!


In [7]:
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

try:
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    if hf_token:
        os.environ["HF_TOKEN"] = hf_token
        login(token=hf_token, add_to_git_credential=True)
        print("SUCCESS: Successfully logged in to Hugging Face!")
    else:
        print("ERROR: HF_TOKEN secret is empty. Please check your Kaggle Secrets.")
except Exception as e:
    print(f"Error logging in: {e}")

Token has not been saved to git credential helper.
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Cannot authenticate through git-credential as no helper is defined on your machine.
You might have to re-authenticate when pushing to the Hugging Face Hub.
Run the following command in your terminal in case you want to set the 'store' credential helper as default.

git config --global credential.helper store

Read https://git-scm.com/book/en/v2/Git-Tools-Credential-Storage for more details.
SUCCESS: Successfully logged in to Hugging Face!


In [13]:
import sys
sys.path.insert(0, "/kaggle/working/indicf5/src")

import os
import re
import subprocess
import tempfile
import numpy as np
import soundfile as sf
import torch
from datasets import load_dataset
from IPython.display import Audio, display

# 1. Imports from our package
from indicf5_finetune.evaluate import load_model_from_checkpoint, load_vocoder, synthesize

# Detect the checkpoint and vocab directories dynamically (prioritize merged working/model_last.pt)
if os.path.exists("/kaggle/working/model_last.pt"):
    checkpoint_dir = "/kaggle/working"
else:
    checkpoint_dir = "/kaggle/input/datasets/ragunathravi/lora-finetuned-indicf5/indicf5/checkpoints"
    if not os.path.exists(checkpoint_dir):
        checkpoint_dir = "/kaggle/working/indicf5/checkpoints"
        if not os.path.exists(checkpoint_dir):
            checkpoint_dir = "./checkpoints"

vocab_path = "/kaggle/input/datasets/ragunathravi/lora-finetuned-indicf5/indicf5/data_tamil/vocab.txt"
if not os.path.exists(vocab_path):
    vocab_path = "/kaggle/working/indicf5/data_tamil/vocab.txt"
    if not os.path.exists(vocab_path):
        vocab_path = "./data_tamil/vocab.txt"

output_dir = "/kaggle/working/eval_output"
os.makedirs(output_dir, exist_ok=True)

print(f"Using checkpoint directory: {checkpoint_dir}")
print(f"Using vocab file: {vocab_path}")

# Helper to run punctuation restoration inside the isolated cadence_env on GPU 1 (or GPU 0 fallback)
def restore_punctuation_isolated(text):
    cadence_python = "/kaggle/working/cadence_env/bin/python"
    if not os.path.exists(cadence_python):
        print("WARNING: Isolated cadence_env python not found. Skipping punctuation restoration.")
        return text
        
    print("Restoring punctuation using Cadence in isolated environment on GPU...")
    # Write text to temp file to prevent command line length limitations
    with tempfile.NamedTemporaryFile(mode='w+', suffix='.txt', delete=False, encoding='utf-8') as f_in, \
         tempfile.NamedTemporaryFile(mode='w+', suffix='.txt', delete=False, encoding='utf-8') as f_out:
        in_path = f_in.name
        out_path = f_out.name
        f_in.write(text)
        f_in.close()
        f_out.close()
        
    try:
        # Python code to run inside the virtualenv (GPU 1 mapping with GPU 0 fallback and paragraph batching)
        punc_code = f'''
import os
import torch
from cadence import PunctuationModel
with open("{in_path}", "r", encoding="utf-8") as f:
    input_text = f.read()

# Map Cadence to GPU 1 if dual-GPU is available, else fallback to GPU 0
gpu_id = 1 if torch.cuda.device_count() > 1 else 0
print(f"Loading Cadence on GPU device: cuda:{{gpu_id}}")

model = PunctuationModel(model="Cadence", gpu_id=gpu_id)

# Split by paragraphs (newlines) to preserve structure and enable batch processing
paragraphs = input_text.split("\\n")
non_empty_indices = [i for i, p in enumerate(paragraphs) if p.strip()]
non_empty_texts = [paragraphs[i] for i in non_empty_indices]

if non_empty_texts:
    punctuated_non_empty = model.punctuate(non_empty_texts, batch_size=8)
    # Map back to original paragraph indices
    for index, p_text in zip(non_empty_indices, punctuated_non_empty):
        paragraphs[index] = p_text

# Re-join paragraphs with newlines
punctuated_text = "\\n".join(paragraphs)

with open("{out_path}", "w", encoding="utf-8") as f:
    f.write(punctuated_text)
'''
        # Run the isolated command
        env = os.environ.copy()
        env["PYTHONNOUSERSITE"] = "1"
        cmd = [cadence_python, "-c", punc_code]
        res = subprocess.run(cmd, capture_output=True, text=True, env=env)
        
        if res.returncode != 0:
            print(f"WARNING: Cadence subprocess returned error: {res.stderr}")
            return text
            
        # Read results back
        with open(out_path, "r", encoding="utf-8") as f:
            punctuated_text = f.read()
        print("SUCCESS: Punctuation restored successfully!")
        return punctuated_text
    except Exception as e:
        print(f"WARNING: Punctuation restoration failed: {e}")
        return text
    finally:
        # Clean up temp files
        for path in [in_path, out_path]:
            if os.path.exists(path):
                os.remove(path)

# 3. Load reference audio and text from local Kaggle Dataset
ref_audio_path = "/kaggle/input/datasets/ragunathravi/savukku-ref-audio/reference.wav"
ref_txt_path = "/kaggle/input/datasets/ragunathravi/savukku-ref-audio/reference.txt"

if not os.path.exists(ref_audio_path):
    ref_audio_path = "./reference.wav"
    ref_txt_path = "./reference.txt"

print(f"\nLoading reference voice from: {ref_audio_path}")

# Read reference transcript text
with open(ref_txt_path, "r", encoding="utf-8") as f: 
    ref_text = f.read().strip()

# Get samplerate of reference audio
ref_audio_info = sf.info(ref_audio_path)
ref_sr = ref_audio_info.samplerate

print(f"  Successfully loaded reference voice!")
print(f"  Reference Transcript: '{ref_text}'")

# 4. Load the fine-tuned model and Vocoder
# Explicitly load IndicF5 on GPU 0
device = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"\nLoading model on {device}...")
model, step = load_model_from_checkpoint(checkpoint_dir, vocab_path, device=device)
vocoder = load_vocoder(vocoder_name="vocos", is_local=False, device=device)

# 5. Define script with custom option fallback (uses custom text if provided, else predefined text)
custom_text = """அட்ரஸ்க்கு ஒரு மெயில் அனுப்பிச்சிட்டு இருக்காங்க ஏன்னா ஒருவேளை அவங்க எங்கயாவது ஒரு
ஐலாண்ட்ல இருந்து ஏதோ ஒரு மிராக்கிள் நடந்து ஒரு ரிப்ளை வந்துராதான்னு நினைச்சு இந்த
அளவுக்கு ஹோப்புல்லா அங்க இருக்கற மக்கள் இருந்துட்டு இருக்காங்க அட் த சேம் டைம் ஹோப்
இல்லாமலும் இருக்காங்க என்னங்க இத்தனை வருஷம் ஆச்சு திருப்பி கிடைச்சிருவாங்களாங்கற
கேள்வியும் இருக்குது. இது வந்து ஒரு மிகப்பெரிய மன உளைச்சல். இந்த மன உளைச்சல்ல
இருக்கறவங்களுக்கு ஒரு கக்ளோஷர் கொடுக்கணும்ங்கறதுக்காகதான். பிரேக்அப்னா அத எண்ட் பண்ணு
"""

if custom_text.strip() and not custom_text.strip().startswith("PASTE YOUR"):
    tamil_text_to_generate = custom_text
    print("\nUsing custom script provided by user.")
else:
    tamil_text_to_generate = (
        '''ஓகே கய்ஸ், இப்போ நம்ப பேச போறது என்னன்னா, டாஸ்மாக் தமிழ்நாட்டுல டாஸ்மாக் எப்படி? இவ்வளவு பெரிய சாம்ராஜ்யத்தை உருவாக்குனாங்க அப்படி என்றத பேசலாம் முதல் விஷயம் என்னன்னா?
 ஃபர்ஸ்ட் பாயிண்ட் நான் என்ன சொல்ல வரேன்னா அவங்க ஃபர்ஸ்ட் இது வந்து உருவாக்கினது வந்து ஜெயலலிதா ஆக்சுவலா இப்ப நம்ம நிறைய வீடியோஸ் பார்க்கிறோம் இல்லையா, சவுக்கு சங்கர் வீடியோ அந்த மாதிரி வீடியோஸ் பார்க்கும்போது அவர் சொல்வார் இல்லையா இது உருவாக்கியது வந்து ஜெயலலிதா ஆக்சுவலா இது வந்து பிரைவேட் கிட்ட தான் இருந்தது பட் கொஞ்ச நாளுக்கு அப்புறம் இது வந்து அரசுடைமை ஆக்குறாங்க அப்பா அப்படி ஆகும் போது டாஸ்மாக் வந்து சில குறிப்பிட்ட நபர்களுக்கு மட்டும் போய் சேர மாதிரி சில விஷயங்கள் சில பில்ஸ் பேசப்படுது. 
'''
    )
    print("\nNo custom script found. Using predefined default script (TASMAC).")

# 6. Restore punctuation using isolated Cadence environment
tamil_text_to_generate = restore_punctuation_isolated(tamil_text_to_generate)

# 7. Segment target text ONLY on major periods/dandas (. or ।) for naturalness
segments = []
for p in tamil_text_to_generate.split("\n\n"):
    p = p.strip()
    if not p:
        continue
    # Split ONLY on major period marks (. or ।) as requested
    s = re.split(r'(?<=[.।])\s+', p)
    current = ""
    for sentence in s:
        if len(current) + len(sentence) < 280:
            current += " " + sentence
        else:
            if current:
                segments.append(current.strip())
            current = sentence
    if current:
        segments.append(current.strip())

print(f"\nSplit target text into {len(segments)} segments for long-form synthesis.")

# 8. Generate audio segment-by-segment and concatenate them
audio_parts = []
for i, seg in enumerate(segments):
    print(f"Generating segment {i+1}/{len(segments)}: '{seg[:45]}...'")
    seg_audio, duration, elapsed, rtf = synthesize(
        model=model,
        vocoder=vocoder,
        text=seg,
        ref_audio=ref_audio_path,
        ref_text=ref_text,
        device=device,
        nfe_step=32
    )
    audio_parts.append(seg_audio)
    
    # Add 0.25 seconds of silence pause between segments for natural transitions
    silence = np.zeros(int(24000 * 0.25), dtype=np.float32)
    audio_parts.append(silence)

audio_data = np.concatenate(audio_parts)

# 9. Save output audio
output_wav_path = os.path.join(output_dir, "tamil_cloned_voice.wav")
sf.write(output_wav_path, audio_data, 24000)
print(f"\nSUCCESS: Generated audio saved to {output_wav_path}")

# 10. Render interactive Audio players in Kaggle
print("\n🔊 Reference Voice (Original):")
display(Audio(ref_audio_path, rate=ref_sr))

print("\n🎉 Generated Voice (Cloned in Tamil):")
display(Audio(output_wav_path, rate=24000))


Using checkpoint directory: /kaggle/input/notebooks/ragunathravi/fork-of-new-indicf5-finetune-with-lora/indicf5/checkpoints
Using vocab file: /kaggle/input/notebooks/ragunathravi/fork-of-new-indicf5-finetune-with-lora/indicf5/data_tamil/vocab.txt

Streaming reference voice from Hugging Face...


Resolving data files:   0%|          | 0/53 [00:00<?, ?it/s]

  Successfully loaded reference voice!
  Reference Transcript: 'இப்படி இப்படி சொல்ற அப்படின்னு யோசிக்க தோணும் அது உண்மையிலேயே நடந்திருந்தா அதை குடிச்சிருந்தா எப்படி இருக்கும்னு யோசிச்சு பாருங்க ஏன்னா நடந்துச்சு அது குடிச்சாங்க சோ ஒரு காமன் டேங்க்ல குடி தண்ணி டேங்க்ல நம்ம எல்லாருமே டேப்பை'

Loading model on cuda...
Loading checkpoint: /kaggle/input/notebooks/ragunathravi/fork-of-new-indicf5-finetune-with-lora/indicf5/checkpoints/model_last.pt
  Loaded checkpoint at step 5532
Download Vocos from huggingface charactr/vocos-mel-24khz

Synthesizing new speech: 'ஓகே கய்ஸ், இப்போ நம்ப பேச போறது என்னன்னா, டாஸ்மாக் தமிழ்நாட்டுல டாஸ்மாக் எப்படி? இவ்வளவு பெரிய சாம்ராஜ்யத்தை உருவாக்குனாங்க அப்படி என்றத பேசலாம் முதல் விஷயம் என்னன்னா?
 ஃபர்ஸ்ட் பாயிண்ட் நான் என்ன சொல்ல வரேன்னா அவங்க ஃபர்ஸ்ட் இது வந்து உருவாக்கினது வந்து ஜெயலலிதா ஆக்சுவலா இப்ப நம்ம நிறைய வீடியோஸ் பார்க்கிறோம் இல்லையா, சவுக்கு சங்கர் வீடியோ அந்த மாதிரி வீடியோஸ் பார்க்கும்போது அவர் சொல்வார் இல்லையா இது உருவாக்கியது வந்து ஜெயலலிதா ஆக்சு


🎉 Generated Voice (Cloned in Tamil):
